# ANN Orchestrator For Multi-Run Ensembling and Checkpointing

This notebook defines a reusable orchestrator that:
1. Configures a set of runs (architectures, hyperparameters, number of seeds).
2. Runs expanding-window forecasts and top-k ensembling (same logic as `compute_top_k_ensemble`).
3. Saves model checkpoints at each refit step (DeepSHAP-friendly: torch `state_dict` + preprocessing state).
4. Saves performance tuples per maturity: `(maturity, r2_oos, rsz_pval)`.

The example at the bottom runs a simple forward-only ANN with `n_models=10` and `k_top=1`.

In [ ]:
import os
import sys
from pathlib import Path
from dataclasses import dataclass, asdict
from datetime import datetime
from typing import Callable, Any

import numpy as np
import pandas as pd
import torch
import statsmodels.api as sm
from scipy.stats import t as tstat
from tqdm.auto import tqdm

sys.path.insert(0, os.path.abspath('..'))

import utils.base_utils as bu
import utils.window_utils as wu
from utils.macro_grouping import add_group_level, build_full_group_mapping
from models.ann_vector_validation import PyTorchMLPWrapper
from models.macro_forward_ann import MacroForwardANNWrapper
from models.group_ensemble_ann import GroupEnsembleANNWrapper

# Data prep --------------------------------------------------------------
# Mirrors notebooks/PCA_PLS_results.ipynb and notebooks/shap/shap_compute.ipynb:
# naive 1-month shift for publication lag (no per-series registry), then drop
# the two FRED series that start late. Keep these three data-prep cells in
# sync — the SHAP runner loads the saved scalers and will reject an X with a
# different column count.

start_date = '1971-08-31'
end_date = '2025-06-30'
# Annual 12m holding returns overlap month-to-month; gap=11 matches init.ipynb GAP=11 for
# SER-style non-overlapping OOS inference (oos_r2 / RSZ Signif).
GAP_ANNUAL_NONOVERLAP = 11

yields = bu.get_yields(type='kr', start=start_date, end=end_date,
                       maturities=[str(i) for i in range(12, 121) if i % 12 == 0])
forward = bu.get_forward_rates(yields)
xr = bu.get_excess_returns(yields, horizon=12).dropna()

fred_md_start_date = pd.to_datetime(start_date) - pd.DateOffset(months=6)
fred_md_raw = bu.get_fred_data('data/2026-01-MD.csv',
                               start=fred_md_start_date, end=end_date)

fred_md = fred_md_raw.shift(1)
fred_md = fred_md.drop(columns=['TWEXAFEGSMTHx', 'ACOGNO'])
fred_md = fred_md[start_date:end_date]

yields = yields.loc[yields.index <= xr.index[-1]]
forward = forward.loc[forward.index <= xr.index[-1]]
xr = xr.loc[xr.index <= xr.index[-1]]
fred_md = fred_md.loc[fred_md.index <= xr.index[-1]]

s2g = build_full_group_mapping(fred_md, forward, yields)
X = pd.concat([fred_md, forward, yields], axis=1,
              keys=['fred', 'forward', 'yields'])
X = add_group_level(X, s2g, level_name='group')
X = X.sort_index(axis=1, level='group')

# Same target construction as notebooks/init.ipynb (literal column list, not xr[maturities]).
y_all = xr[['24', '36', '48', '60', '72', '84', '96', '108', '120']].values
# One label per column of y_all (order must match) — used only for RunConfig performance rows.
maturities = ['24', '36', '48', '60', '72', '84', '96', '108', '120']
dates = xr.index
print('X shape:', X.shape, '| y_all shape:', y_all.shape,
      '| dates:', dates[0].date(), '->', dates[-1].date())

X shape: (635, 144) | y_all shape: (635, 9) | dates: 1971-08-31 -> 2024-06-30


/home/ulrikts/Documents/NTNU/TIO4900-Replication/utils/macro_grouping.py:219: UserWarning: The following series are defined in get_fredmd_grouping() but are not present in the FRED-MD data: ['ACOGNO', 'TWEXAFEGSMTHx']. They may have been dropped or renamed.
  warnings.warn(
/home/ulrikts/Documents/NTNU/TIO4900-Replication/utils/macro_grouping.py:168: UserWarning: 2 entries in series_to_group are not present in the DataFrame columns: ['ACOGNO', 'TWEXAFEGSMTHx']
  warnings.warn(


In [3]:
from utils.orchestrator_runs import RunConfig, run_experiment

Main block for running:

In [ ]:
nodes = [3, 5]
architectures = [(h,) for h in nodes] + [(h, h) for h in nodes]

def arch_to_name(arch):
    return "&".join(str(x) for x in arch)

def make_fwd_ann_cfg(arch, n_models=100, k_top=10):
    return RunConfig(
        run_name=f"fwd_ann_{arch_to_name(arch)}_{n_models}runs_top{k_top}",
        model_builder=lambda seed, arch=arch: PyTorchMLPWrapper(
        archi=arch,
        lr=0.01,
        epochs=1000,
        tune_every=60,
        patience=50,
        param_grid={"penalty": [0.01, 0.001, 0.0001]},
        seed=seed,
        use_pca=False,
        n_components=None,
        y_center=False,
        ),
        n_models=n_models,
        k_top=k_top,
        maturities=maturities,
        oos_start=pd.Timestamp("1990-01-31"),
        gap=0,
        refit_freq=1,
        benchmark="hist_mean",
        progress=True,
        )

run_configs = [make_fwd_ann_cfg(arch, n_models = 100, k_top = 10) for arch in architectures]

all_summaries = []
for cfg in run_configs:
    print(f"Running {cfg.run_name} ...")
    all_summaries.append(run_experiment(cfg, X, y_all, dates))

Running fwd_ann_3_5runs_top2 ...


Seeds:  60%|██████    | 3/5 [01:21<00:54, 27.10s/it]


KeyboardInterrupt: 

In [ ]:
def arch_to_name(arch):
    return "&".join(str(x) for x in arch)


# Macro–forward ANN: notebooks/init.ipynb MacroForwardANN with archi_macro=(32,) —
# e.g. MacroForwardANNWrapper(archi_forward=(7,7,7), archi_macro=(32,), ...,
# param_grid={'penalty': [1e-4, 1e-3, 0.01, 0.1, 1.0], 'dropout_rate': [0.0, 0.1]}.
ANN_PARAM_GRID_INIT_MACRO_FORWARD = {
    "penalty": [1e-4, 1e-3, 0.01, 0.1, 1.0],
    "dropout_rate": [0.0, 0.1],
}

# Group–ensemble ANN: notebooks/init.ipynb commented GroupEnsembleANNWrapper lines
# (param_grid={'penalty': [0.001, 0.0001], 'dropout_rate': [0.0, 0.1, 0.3]}, ...).
ANN_PARAM_GRID_INIT_GROUP_ENSEMBLE = {
    "penalty": [0.001, 0.0001],
    "dropout_rate": [0.0, 0.1, 0.3],
}


def print_oos_r2_summary(run_label, performance_tuples):
    """Per-maturity R²_oos like notebooks/init.ipynb (--- OOS R² Summary --- / Nm lines)."""
    print(f"\n--- OOS R² Summary for {run_label} ---")
    for mat, r2, _pv in performance_tuples:
        print(f"  {mat}m: {r2:.4f}")


def make_macro_forward_ann_cfg(
    archi_forward=(3, 3, 3),
    archi_macro=(32,),
    gap=GAP_ANNUAL_NONOVERLAP,
    n_models=100,
    k_top=10,
):
    """Macro–forward ANN: forward-rate tower + macro (FRED) tower.

    Default architecture: forward tower (3-3-3), macro tower (32). OOS uses
    gap=GAP_ANNUAL_NONOVERLAP (11) for non-overlapping annual 12m returns (init GAP=11).
    """
    return RunConfig(
        run_name=(
            f"macro_forward_ann_gap{gap}_fwd{arch_to_name(archi_forward)}_"
            f"macro{arch_to_name(archi_macro)}_{n_models}runs_top{k_top}"
        ),
        model_builder=lambda seed, fwd=archi_forward, mac=archi_macro: MacroForwardANNWrapper(
            archi_forward=fwd,
            archi_macro=mac,
            lr=0.001,
            epochs=1000,
            tune_every=60,
            patience=50,
            param_grid=ANN_PARAM_GRID_INIT_MACRO_FORWARD,
            seed=seed,
            y_center=True,
            activation="tanh",
        ),
        n_models=n_models,
        k_top=k_top,
        maturities=maturities,
        oos_start=pd.Timestamp("1990-01-31"),
        gap=gap,
        refit_freq=1,
        benchmark="hist_mean",
        progress=True,
    )


def make_group_ensemble_ann_cfg(
    archi_forward=(3, 3),
    archi_macro=(1,),  # one hidden node per FRED group's tower (shared shape across groups)
    gap=GAP_ANNUAL_NONOVERLAP,
    n_models=100,
    k_top=10,
):
    """Group–ensemble ANN: one tower per FRED-MD category + forward tower (archi_forward)."""
    return RunConfig(
        run_name=(
            f"group_ensemble_ann_gap{gap}_fwd{arch_to_name(archi_forward)}_"
            f"gmacro{arch_to_name(archi_macro)}_{n_models}runs_top{k_top}"
        ),
        model_builder=lambda seed, fwd=archi_forward, mac=archi_macro: GroupEnsembleANNWrapper(
            archi_forward=fwd,
            archi_macro=mac,
            lr=0.001,
            epochs=1000,
            tune_every=60,
            patience=50,
            param_grid=ANN_PARAM_GRID_INIT_GROUP_ENSEMBLE,
            seed=seed,
            y_center=True,
            activation="tanh",
        ),
        n_models=n_models,
        k_top=k_top,
        maturities=maturities,
        oos_start=pd.Timestamp("1990-01-31"),
        gap=gap,
        refit_freq=1,
        benchmark="hist_mean",
        progress=True,
    )

In [ ]:
# Macro–forward ANN + group–ensemble ANN (100 seeds, top–10 ensemble, gap=11).
run_configs = [
    make_macro_forward_ann_cfg(),
    make_group_ensemble_ann_cfg(),
]

all_summaries = []
for cfg in run_configs:
    print(f"Running {cfg.run_name} ...")
    summary = run_experiment(cfg, X, y_all, dates)
    all_summaries.append(summary)
    print_oos_r2_summary(cfg.run_name, summary["performance"])
    print("Artifacts:", summary["run_dir"], "\n")

Seeds:   0%|          | 0/5 [00:00<?, ?it/s]

Running macro_forward_ann_3_5runs_top2 ...


Seeds:   0%|          | 0/5 [00:00<?, ?it/s]

Running macro_forward_ann_5_5runs_top2 ...


Seeds:   0%|          | 0/5 [00:00<?, ?it/s]

Running macro_forward_ann_3&3_5runs_top2 ...


Seeds:   0%|          | 0/5 [00:00<?, ?it/s]

Running macro_forward_ann_5&5_5runs_top2 ...


Seeds:  40%|████      | 2/5 [03:33<05:41, 113.88s/it]

In [5]:
# ----------------------------------------------------------------------
# SHAP-oriented mini run: fewer seeds / k. Same arches as macro + group mains;
# param grids = init MacroForward (macro-32 style) vs init commented GroupEnsemble.
# ----------------------------------------------------------------------
VALID_MACRO_FWD = (3, 3, 3)
VALID_MACRO_MAC = (32,)
VALID_GRP_FWD = (3, 3)
VALID_GRP_GMACRO = (1,)
VALID_SEEDS = 5
VALID_KTOP = 2
VALID_OOS_START = pd.Timestamp("1990-01-31")
VALID_MACRO_GRIDDED = {"penalty": [1e-4, 1e-3, 0.01, 0.1, 1.0], "dropout_rate": [0.0, 0.1]}
VALID_GROUP_GRIDDED = {"penalty": [0.001, 0.0001], "dropout_rate": [0.0, 0.1, 0.3]}


def make_macro_forward_ann_valid_cfg():
    return RunConfig(
        run_name=(
            f"macro_forward_ann_valid_gap{GAP_ANNUAL_NONOVERLAP}_"
            f"fwd{arch_to_name(VALID_MACRO_FWD)}_macro{arch_to_name(VALID_MACRO_MAC)}_"
            f"{VALID_SEEDS}seeds"
        ),
        model_builder=lambda seed: MacroForwardANNWrapper(
            archi_forward=VALID_MACRO_FWD,
            archi_macro=VALID_MACRO_MAC,
            lr=0.001,
            epochs=1000,
            tune_every=60,
            patience=50,
            param_grid=VALID_MACRO_GRIDDED,
            seed=seed,
            y_center=True,
            activation="tanh",
        ),
        n_models=VALID_SEEDS,
        k_top=VALID_KTOP,
        maturities=maturities,
        oos_start=VALID_OOS_START,
        gap=GAP_ANNUAL_NONOVERLAP,
        refit_freq=1,
        benchmark="hist_mean",
        progress=True,
    )


def make_group_ensemble_ann_valid_cfg():
    return RunConfig(
        run_name=(
            f"group_ensemble_ann_valid_gap{GAP_ANNUAL_NONOVERLAP}_"
            f"fwd{arch_to_name(VALID_GRP_FWD)}_gmacro{arch_to_name(VALID_GRP_GMACRO)}_{VALID_SEEDS}seeds"
        ),
        model_builder=lambda seed: GroupEnsembleANNWrapper(
            archi_forward=VALID_GRP_FWD,
            archi_macro=VALID_GRP_GMACRO,
            lr=0.001,
            epochs=1000,
            tune_every=60,
            patience=50,
            param_grid=VALID_GROUP_GRIDDED,
            seed=seed,
            y_center=True,
            activation="tanh",
        ),
        n_models=VALID_SEEDS,
        k_top=VALID_KTOP,
        maturities=maturities,
        oos_start=VALID_OOS_START,
        gap=GAP_ANNUAL_NONOVERLAP,
        refit_freq=1,
        benchmark="hist_mean",
        progress=True,
    )


valid_configs = [
    make_macro_forward_ann_valid_cfg(),
    make_group_ensemble_ann_valid_cfg(),
]

valid_summaries = []
for cfg in valid_configs:
    print(f"Running {cfg.run_name} ...")
    summary = run_experiment(cfg, X, y_all, dates)
    valid_summaries.append(summary)
    print_oos_r2_summary(cfg.run_name, summary["performance"])
    print("Artifacts:", summary["run_dir"], "\n")

Running macro_forward_ann_valid_3_5seeds ...


Seeds: 100%|██████████| 5/5 [03:36<00:00, 43.37s/it]


Running group_ensemble_ann_valid_fwd3_grp3_5seeds ...


Seeds: 100%|██████████| 5/5 [11:58<00:00, 143.71s/it]

/Users/trineberntsensaether/Documents/Master thesis/master_code/TIO4900-Replication/artifacts/orchestrator_runs/macro_forward_ann_valid_3_5seeds/20260422_103724 -> [('24', 0.11215451170664925, 0.0021511147422829113), ('36', 0.1646069339459798, 0.00034351844664581854), ('48', 0.2150142106877112, 0.00029067197578047654), ('60', 0.18765969706465402, 8.746671254333815e-05), ('84', 0.1905547141933519, 0.0001691447469853724), ('120', 0.2406432851144239, 5.9441925054692923e-05)]
/Users/trineberntsensaether/Documents/Master thesis/master_code/TIO4900-Replication/artifacts/orchestrator_runs/group_ensemble_ann_valid_fwd3_grp3_5seeds/20260422_104101 -> [('24', 0.005484476073984168, 7.991032202170345e-06), ('36', 0.1800257529190783, 1.5139043338940894e-06), ('48', 0.15380648757318283, 6.053871732802918e-07), ('60', 0.09135095351705502, 5.268591939389111e-07), ('84', 0.1769171475910155, 1.7245142880373265e-07), ('120', 0.18612212738666545, 3.6715822382404895e-07)]


## SHAP lives elsewhere

This notebook only trains models and saves checkpoints. SHAP values are computed in `notebooks/shap/shap_compute.ipynb`, which loads the checkpoints produced here and uses the modular adapters under `utils/shap_adapters/`.